# 🧞 Doc-Genie v2.0
### Professional Python Docstring Generator
> **IBM Granite 3.3 2B Instruct** × **HuggingFace** × **Gradio** × **AST Analysis** × **ReportLab PDF**

---
**Steps:**
1. Run **Cell 1** — Install dependencies
2. Run **Cell 2** — (Optional) HuggingFace login
3. Run **Cell 3** — Launch Doc-Genie
4. Click the **public Gradio URL** printed in the output

> ⚡ **Tip:** Go to `Runtime → Change runtime type → T4 GPU` for faster generation!

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 1 — Install All Dependencies           ║
# ║  Run this FIRST. Takes ~2-3 minutes.         ║
# ╚══════════════════════════════════════════════╝

!pip install -q gradio>=4.0.0 \
               transformers>=4.40.0 \
               accelerate>=0.30.0 \
               torch \
               sentencepiece \
               huggingface_hub>=0.22.0 \
               reportlab>=4.0.0 \
               bitsandbytes

print('\n✅ All dependencies installed successfully!')

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 2 — HuggingFace Login (Optional)       ║
# ║  Only needed if the model requires auth.     ║
# ╚══════════════════════════════════════════════╝

# Uncomment and paste your token if needed:
# from huggingface_hub import login
# login('hf_YOUR_TOKEN_HERE')

print('Skipped (no token needed for public model).')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Doc-Genie v2.0 Full Application                               ║
# ║  IBM Granite 3.3 2B Instruct × AST Analysis × Gradio × ReportLab PDF    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import ast
import re
import os
import time
import textwrap
import traceback
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable, KeepTogether, PageBreak
)

print('✅ Imports OK')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


# ════════════════════════════════════════════════════════════
#  CONFIG
# ════════════════════════════════════════════════════════════
MODEL_ID         = 'ibm-granite/granite-3.3-2b-instruct'
MAX_SOURCE_LINES = 80
VERSION          = '2.0'


# ════════════════════════════════════════════════════════════
#  1. MODEL MANAGER
# ════════════════════════════════════════════════════════════
class ModelManager:
    """Singleton: loads Granite 3.3 2B Instruct once and caches it."""

    _tokenizer = None
    _model     = None
    _loaded    = False

    @classmethod
    def load(cls):
        if cls._loaded:
            return cls._tokenizer, cls._model
        print(f'\n⏳ Loading {MODEL_ID}...')
        print('   (First run downloads ~5 GB — please wait)')
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        cls._tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
        cls._model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=dtype,
            device_map='auto',
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
        cls._model.eval()
        cls._loaded = True
        print('✅ Model loaded!')
        return cls._tokenizer, cls._model

    @classmethod
    def infer(cls, prompt, max_new_tokens=512, temperature=0.35, top_p=0.90):
        tok, mdl = cls.load()
        inputs   = tok(prompt, return_tensors='pt').to(mdl.device)
        in_len   = inputs['input_ids'].shape[1]
        with torch.no_grad():
            out = mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                repetition_penalty=1.18,
                pad_token_id=tok.eos_token_id,
                eos_token_id=tok.eos_token_id,
            )
        return tok.decode(out[0][in_len:], skip_special_tokens=True)


# ════════════════════════════════════════════════════════════
#  2. AST ANALYSER
# ════════════════════════════════════════════════════════════
class ASTAnalyser:
    """Deep static analysis via Python ast module."""

    IO_NAMES = frozenset({
        'open','read','write','readline','readlines','close','flush',
        'seek','tell','print','input','urlopen','urlretrieve',
        'requests','httpx','aiohttp','boto3','sqlite3','psycopg2',
        'pymongo','redis','send','recv','connect','listen',
    })

    @staticmethod
    def parse(source):
        try:
            tree = ast.parse(source)
        except SyntaxError as exc:
            return [{'error': f'{exc.msg} (line {exc.lineno})'}]
        results = []
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                try:
                    results.append(ASTAnalyser._extract(node, source))
                except Exception as exc:
                    results.append({'error': str(exc), 'name': getattr(node, 'name', '?')})
        return results

    @staticmethod
    def _up(node):
        try: return ast.unparse(node)
        except: return ''

    @classmethod
    def _extract(cls, node, source):
        src_lines = source.splitlines()
        start = node.lineno - 1
        end   = min(getattr(node, 'end_lineno', start + MAX_SOURCE_LINES), start + MAX_SOURCE_LINES)
        info  = {
            'name': node.name, 'is_async': isinstance(node, ast.AsyncFunctionDef),
            'lineno': node.lineno, 'decorators': [], 'args': [],
            'return_type': None, 'raises': [], 'calls': [], 'assigned_vars': [],
            'has_loops': False, 'has_io': False, 'has_yield': False,
            'has_return': False, 'complexity': 1,
            'body_source': '\n'.join(src_lines[start:end]),
            'existing_doc': ast.get_docstring(node) or '',
        }
        for d in node.decorator_list:
            info['decorators'].append(cls._up(d))
        if node.returns:
            info['return_type'] = cls._up(node.returns)
        ad = node.args
        all_pos = ad.posonlyargs + ad.args
        offset  = len(all_pos) - len(ad.defaults)
        for i, arg in enumerate(all_pos):
            kind = 'positional-only' if i < len(ad.posonlyargs) else 'positional'
            di   = i - offset
            default = cls._up(ad.defaults[di]) if 0 <= di < len(ad.defaults) else None
            info['args'].append({'name': arg.arg,
                                 'type': cls._up(arg.annotation) if arg.annotation else None,
                                 'default': default, 'kind': kind})
        for i, arg in enumerate(ad.kwonlyargs):
            kd = ad.kw_defaults[i]
            info['args'].append({'name': arg.arg,
                                 'type': cls._up(arg.annotation) if arg.annotation else None,
                                 'default': cls._up(kd) if kd else None, 'kind': 'keyword-only'})
        if ad.vararg:
            info['args'].append({'name': f'*{ad.vararg.arg}',
                                 'type': cls._up(ad.vararg.annotation) if ad.vararg.annotation else None,
                                 'default': None, 'kind': 'var-positional'})
        if ad.kwarg:
            info['args'].append({'name': f'**{ad.kwarg.arg}',
                                 'type': cls._up(ad.kwarg.annotation) if ad.kwarg.annotation else None,
                                 'default': None, 'kind': 'var-keyword'})
        for child in ast.walk(node):
            if isinstance(child, (ast.For, ast.While)):
                info['has_loops'] = True; info['complexity'] += 1
            if isinstance(child, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
                info['has_loops'] = True
            if isinstance(child, (ast.If, ast.IfExp)): info['complexity'] += 1
            if isinstance(child, ast.ExceptHandler):    info['complexity'] += 1
            if isinstance(child, (ast.With, ast.AsyncWith)): info['complexity'] += 1
            if isinstance(child, ast.Raise):
                info['raises'].append(cls._up(child.exc) if child.exc else 'Exception')
            if isinstance(child, (ast.Yield, ast.YieldFrom)): info['has_yield'] = True
            if isinstance(child, ast.Return) and child.value: info['has_return'] = True
            if isinstance(child, ast.Assign):
                for t in child.targets:
                    if isinstance(t, ast.Name): info['assigned_vars'].append(t.id)
            if isinstance(child, ast.Call):
                fn = ''
                if isinstance(child.func, ast.Attribute): fn = cls._up(child.func)
                elif isinstance(child.func, ast.Name):    fn = child.func.id
                if fn:
                    info['calls'].append(fn)
                    if fn.split('.')[0] in cls.IO_NAMES or any(k in fn for k in cls.IO_NAMES):
                        info['has_io'] = True
        info['raises'] = list(dict.fromkeys(info['raises']))
        info['calls']  = list(dict.fromkeys(info['calls']))[:12]
        return info


# ════════════════════════════════════════════════════════════
#  3. PROMPT BUILDER
# ════════════════════════════════════════════════════════════
class PromptBuilder:
    """Structured prompts for Granite 3.3 chat format."""

    STYLES = {
        'google': (
            'Google Python Style docstring:\n'
            '  1. One-line summary (imperative mood, <=80 chars)\n'
            '  2. Extended description (if logic warrants it)\n'
            '  3. Args:  (name (type): description.)\n'
            '  4. Returns: (type: description)\n'
            '  5. Raises: (ExceptionType: when raised)\n'
            '  6. Example: (>>> runnable code)\n'
        ),
        'numpy': (
            'NumPy/SciPy Style docstring:\n'
            '  1. One-line summary (imperative mood, <=80 chars)\n'
            '  2. Extended description\n'
            '  3. Parameters\n  ----------\n  name : type\n      description.\n'
            '  4. Returns\n  -------\n  type\n      description.\n'
            '  5. Raises\n  ------\n  ExceptionType\n      When raised.\n'
            '  6. Examples\n  --------\n  >>> code\n'
        ),
    }

    @classmethod
    def build(cls, info, style='google'):
        sh   = cls.STYLES.get(style.lower(), cls.STYLES['google'])
        params = '\n'.join(
            f"  {a['name']} ({a['type'] or 'untyped'})"
            + (f", default={a['default']}" if a['default'] is not None else '')
            + f"  [{a['kind']}]"
            for a in info['args']
        ) or '  (no parameters)'

        return (
            '<|system|>\n'
            'You are an expert Python documentation engineer. '
            'Write precise, professional, production-quality docstrings.\n'
            '<|user|>\n'
            f'Write a {sh}\n'
            'Output ONLY the docstring body (no triple-quotes, no ```python fences).\n\n'
            '=== STATIC ANALYSIS ===\n'
            f"Function Name       : {info['name']}\n"
            f"Asynchronous        : {info['is_async']}\n"
            f"Generator (yield)   : {info['has_yield']}\n"
            f"Decorators          : {', '.join(info['decorators']) or 'none'}\n"
            f"Return Annotation   : {info['return_type'] or 'not annotated'}\n"
            f"Has Explicit Return : {info['has_return']}\n"
            f"Raises              : {', '.join(info['raises']) or 'none detected'}\n"
            f"Uses I/O            : {info['has_io']}\n"
            f"Has Loops           : {info['has_loops']}\n"
            f"Cyclomatic Complexity: {info['complexity']}\n"
            f"Key Calls           : {', '.join(info['calls']) or 'none'}\n"
            f"\n=== PARAMETERS ===\n{params}\n"
            f"\n=== SOURCE CODE ===\n{info['body_source']}\n"
            f"\n=== EXISTING DOCSTRING ===\n"
            f"{info['existing_doc'] or '(none — write a new one)'}\n"
            '\n<|assistant|>\n'
        )


# ════════════════════════════════════════════════════════════
#  4. DOCSTRING GENERATOR
# ════════════════════════════════════════════════════════════
class DocstringGenerator:
    """Full pipeline: source -> AST -> prompt -> LLM -> clean -> inject."""

    @staticmethod
    def _clean(raw, func_name):
        raw = re.sub(r'```[a-z]*', '', raw, flags=re.IGNORECASE)
        raw = raw.replace('"""', '').replace("'''", '')
        if '<|' in raw: raw = raw[:raw.index('<|')]
        if f'def {func_name}' in raw: raw = raw[:raw.index(f'def {func_name}')]
        raw = raw.strip() or f'TODO: Add docstring for {func_name}.'
        return f'    """\n{textwrap.indent(raw, "    ")}\n    """'

    @staticmethod
    def _inject(source, func_name, docstring):
        pattern = (
            r'((?:async\s+)?def\s+' + re.escape(func_name) +
            r'\s*\([^)]*\)\s*(?:->[^:]+)?\s*:)'
        )
        done = [False]
        def rep(m):
            if not done[0]:
                done[0] = True
                return m.group(0) + '\n' + docstring
            return m.group(0)
        return re.sub(pattern, rep, source, flags=re.DOTALL)

    def generate(self, source, style='google', max_tokens=480, temperature=0.35):
        functions = ASTAnalyser.parse(source)
        if not functions:
            return source, 'No Python functions detected.', []
        if len(functions) == 1 and 'error' in functions[0]:
            return source, f"**SyntaxError**: {functions[0]['error']}", []

        annotated = source
        reports   = []
        analysis  = []

        for info in functions:
            if 'error' in info:
                reports.append(f"**`{info.get('name','?')}`** — parse error: {info['error']}")
                continue
            try:
                prompt    = PromptBuilder.build(info, style)
                raw       = ModelManager.infer(prompt, max_tokens, temperature)
                docstring = self._clean(raw, info['name'])
                annotated = self._inject(annotated, info['name'], docstring)
                kind = 'async ' if info['is_async'] else ('gen ' if info['has_yield'] else '')
                reports.append(
                    f"✅ **`{kind}{info['name']}`** | "
                    f"complexity={info['complexity']} | "
                    f"{len(info['args'])} param(s) | "
                    f"raises: `{', '.join(info['raises']) or 'none'}`"
                )
                analysis.append({**info, 'generated_docstring': docstring})
            except Exception as exc:
                reports.append(f"❌ **`{info.get('name','?')}`** — Error: {exc}")

        summary = (
            f"**{len(analysis)} function(s) documented** | "
            f"Style: **{style.capitalize()}** | "
            f"Model: `{MODEL_ID.split('/')[-1]}`\n\n" +
            '\n\n'.join(reports)
        )
        return annotated, summary, analysis


# ════════════════════════════════════════════════════════════
#  5. PDF EXPORTER
# ════════════════════════════════════════════════════════════
class PDFExporter:
    NAVY   = colors.HexColor('#0D2137')
    ACCENT = colors.HexColor('#2E86C1')
    STRIPE = colors.HexColor('#F0F5FB')
    LIGHT  = colors.HexColor('#EAF2FF')
    BG_C   = colors.HexColor('#F8F9FA')
    BG_DS  = colors.HexColor('#FEFDF0')
    GREY   = colors.HexColor('#888888')

    @classmethod
    def _st(cls):
        SS = getSampleStyleSheet()
        def ps(n, **kw): return ParagraphStyle(n, parent=SS['Normal'], **kw)
        return {
            'title': ps('T', fontSize=22, textColor=cls.NAVY, fontName='Helvetica-Bold',
                        spaceAfter=2, alignment=TA_CENTER),
            'sub':   ps('S', fontSize=9,  textColor=cls.GREY, spaceAfter=2, alignment=TA_CENTER),
            'h2':    ps('H2', fontSize=13, textColor=cls.NAVY, fontName='Helvetica-Bold',
                        spaceAfter=4, spaceBefore=10),
            'h3':    ps('H3', fontSize=10, textColor=colors.HexColor('#1A5276'),
                        fontName='Helvetica-Bold', spaceAfter=3, spaceBefore=6),
            'body':  ps('B',  fontSize=9, leading=13, spaceAfter=4),
            'code':  ps('C',  fontSize=7, leading=10, fontName='Courier',
                        backColor=cls.BG_C, borderPadding=(4,6,4,6), spaceAfter=2),
            'ds':    ps('DS', fontSize=7.5, leading=11, fontName='Courier',
                        backColor=cls.BG_DS, borderPadding=(5,7,5,7),
                        textColor=colors.HexColor('#4A235A'), spaceAfter=6),
        }

    @classmethod
    def export(cls, original_src, annotated_src, analysis, style,
               path='/tmp/doc_genie_report.pdf'):
        doc = SimpleDocTemplate(path, pagesize=A4,
                                leftMargin=0.72*inch, rightMargin=0.72*inch,
                                topMargin=0.75*inch,  bottomMargin=0.75*inch,
                                title='Doc-Genie v2.0 Report')
        ST    = cls._st()
        story = []

        story += [
            Spacer(1, 0.1*inch),
            Paragraph('Doc-Genie v2.0 Docstring Report', ST['title']),
            Paragraph('Professional Python Documentation Generator', ST['sub']),
            Paragraph(
                f"Generated: {datetime.now().strftime('%B %d, %Y  %H:%M')}  |  "
                f"Style: <b>{style.capitalize()}</b>  |  Functions: <b>{len(analysis)}</b>",
                ST['sub']),
            Spacer(1, 0.06*inch),
            HRFlowable(width='100%', thickness=2, color=cls.NAVY, spaceAfter=14),
        ]

        def brow(k, v):
            bst = ParagraphStyle('KB', parent=ST['body'], fontName='Helvetica-Bold', fontSize=8)
            return [Paragraph(k, bst), Paragraph(str(v), ST['body'])]

        for fn in analysis:
            fn_hdr = Paragraph(
                f"def {fn['name']}()"
                + (' [async]'     if fn['is_async']  else '')
                + (' [generator]' if fn['has_yield'] else ''),
                ST['h2'])

            th = ParagraphStyle('TH', parent=ST['body'], fontName='Helvetica-Bold',
                                fontSize=8, textColor=colors.white)
            tdata = [
                [Paragraph('Property', th), Paragraph('Value', th)],
                brow('Cyclomatic Complexity', fn['complexity']),
                brow('Return Type', fn['return_type'] or 'not annotated'),
                brow('Raises',      ', '.join(fn['raises']) or 'none'),
                brow('Uses I/O',    'Yes' if fn['has_io']    else 'No'),
                brow('Has Loops',   'Yes' if fn['has_loops'] else 'No'),
                brow('Is Generator','Yes' if fn['has_yield'] else 'No'),
                brow('Decorators',  ', '.join(fn['decorators']) or 'none'),
                brow('Line Number', str(fn['lineno'])),
            ]
            tbl = Table(tdata, colWidths=[2.1*inch, 4.4*inch])
            tbl.setStyle(TableStyle([
                ('BACKGROUND',     (0,0),(-1,0), cls.NAVY),
                ('FONTSIZE',       (0,0),(-1,-1), 8),
                ('GRID',           (0,0),(-1,-1), 0.3, colors.HexColor('#CCCCCC')),
                ('ROWBACKGROUNDS', (0,1),(-1,-1), [colors.white, cls.STRIPE]),
                ('TOPPADDING',     (0,0),(-1,-1), 3),
                ('BOTTOMPADDING',  (0,0),(-1,-1), 3),
                ('LEFTPADDING',    (0,0),(-1,-1), 5),
            ]))

            ph = ParagraphStyle('PH', parent=ST['body'], fontName='Helvetica-Bold',
                                fontSize=8, textColor=colors.white)
            prows = [[Paragraph(h, ph) for h in ['Name','Type','Default','Kind']]]
            for p in fn['args']:
                prows.append([Paragraph(p['name'], ST['body']),
                              Paragraph(p['type'] or '-', ST['body']),
                              Paragraph(p['default'] or '-', ST['body']),
                              Paragraph(p['kind'], ST['body'])])
            ptbl = Table(prows, colWidths=[1.5*inch,1.6*inch,1.5*inch,1.9*inch])
            ptbl.setStyle(TableStyle([
                ('BACKGROUND',     (0,0),(-1,0), cls.ACCENT),
                ('FONTSIZE',       (0,0),(-1,-1), 8),
                ('GRID',           (0,0),(-1,-1), 0.3, colors.HexColor('#CCCCCC')),
                ('ROWBACKGROUNDS', (0,1),(-1,-1), [colors.white, cls.LIGHT]),
                ('TOPPADDING',     (0,0),(-1,-1), 3),
                ('BOTTOMPADDING',  (0,0),(-1,-1), 3),
                ('LEFTPADDING',    (0,0),(-1,-1), 5),
            ]))

            ds_safe = (fn['generated_docstring']
                       .replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
                       .replace('\n','<br/>'))

            story.append(KeepTogether([
                fn_hdr,
                HRFlowable(width='100%', thickness=0.5, color=cls.ACCENT, spaceAfter=6),
                Paragraph('Static Analysis', ST['h3']), tbl, Spacer(1,6),
                Paragraph('Parameters',      ST['h3']),
                ptbl if fn['args'] else Paragraph('(no parameters)', ST['body']),
                Spacer(1,6),
                Paragraph('Generated Docstring', ST['h3']),
                Paragraph(f'<font name="Courier" size="7.5">{ds_safe}</font>', ST['ds']),
                Spacer(1,10),
            ]))

        story += [
            PageBreak(),
            Paragraph('Full Annotated Source Code', ST['h2']),
            HRFlowable(width='100%', thickness=1, color=cls.NAVY, spaceAfter=8),
        ]
        cs = ParagraphStyle('FC', parent=ST['code'], fontSize=7, leading=10, fontName='Courier')
        for line in annotated_src.split('\n'):
            safe = line.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
            story.append(Paragraph(safe or ' ', cs))

        story += [
            Spacer(1, 0.2*inch),
            HRFlowable(width='100%', thickness=0.5, color=cls.GREY, spaceAfter=4),
            Paragraph(
                f'Doc-Genie v2.0  |  IBM Granite 3.3 2B  |  {datetime.now().strftime("%Y-%m-%d")}',
                ParagraphStyle('foot', parent=ST['sub'], fontSize=7)),
        ]

        doc.build(story)
        return path


# ════════════════════════════════════════════════════════════
#  6. DEMO CODE
# ════════════════════════════════════════════════════════════
DEMO_CODE = '''\
import math, os, asyncio
from typing import List, Optional, Dict


def calculate_statistics(
    data: List[float],
    weights: Optional[List[float]] = None,
    ddof: int = 0,
) -> Dict[str, float]:
    if not data:
        raise ValueError("data must be a non-empty list")
    if weights and len(weights) != len(data):
        raise ValueError("weights length must match data")
    n = len(data)
    mean = sum(data) / n if not weights else sum(v*w for v,w in zip(data,weights)) / sum(weights)
    variance = sum((x - mean)**2 for x in data) / (n - ddof)
    s = sorted(data)
    median = s[n//2] if n%2 else (s[n//2-1]+s[n//2])/2
    return {"mean": mean, "median": median, "std_dev": math.sqrt(variance),
            "variance": variance, "min": min(data), "max": max(data), "n": n}


async def fetch_user_profile(user_id: int, api_key: str, retries: int = 3) -> Dict:
    if not isinstance(user_id, int) or user_id <= 0:
        raise ValueError(f"Invalid user_id: {user_id}")
    if not api_key:
        raise PermissionError("api_key is required")
    for attempt in range(retries):
        await asyncio.sleep(0.05 * (attempt + 1))
        if attempt == retries - 1:
            return {"id": user_id, "name": "Alice", "role": "admin"}
    raise TimeoutError(f"Failed after {retries} retries")


def merge_sorted_arrays(arr1: List[int], arr2: List[int]) -> List[int]:
    result, i, j = [], 0, 0
    while i < len(arr1) and j < len(arr2):
        if arr1[i] <= arr2[j]: result.append(arr1[i]); i += 1
        else:                  result.append(arr2[j]); j += 1
    return result + arr1[i:] + arr2[j:]


def read_csv(filepath: str, delimiter: str = ",", encoding: str = "utf-8") -> List[Dict]:
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    with open(filepath, "r", encoding=encoding) as fh:
        lines = fh.readlines()
    header = lines[0].strip().split(delimiter)
    return [dict(zip(header, l.strip().split(delimiter))) for l in lines[1:]]


def fibonacci_generator(n: int):
    if n < 0: raise ValueError("n must be non-negative")
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b
'''


# ════════════════════════════════════════════════════════════
#  7. GRADIO CALLBACKS
# ════════════════════════════════════════════════════════════
_gen_instance = None

def _get_gen():
    global _gen_instance
    if _gen_instance is None:
        _gen_instance = DocstringGenerator()
    return _gen_instance


def run_docgenie(code, style, max_tokens, temperature, progress=gr.Progress(track_tqdm=True)):
    if not code or not code.strip():
        return '', 'Please enter some Python code.', None

    progress(0.05, desc='Analysing AST...')
    gen = _get_gen()
    progress(0.15, desc='Loading model (first run downloads ~5GB)...')
    t0 = time.time()

    try:
        progress(0.20, desc='Generating docstrings...')
        annotated, summary, analysis = gen.generate(code, style.lower(), int(max_tokens), float(temperature))
    except Exception as exc:
        return '', f'**Error**: {exc}\n```\n{traceback.format_exc()}\n```', None

    elapsed = time.time() - t0
    progress(0.85, desc='Building PDF...')
    pdf_path = None
    if analysis:
        try:
            pdf_path = PDFExporter.export(code, annotated, analysis, style)
        except Exception as e:
            summary += f'\n\nPDF export failed: {e}'

    progress(1.0, desc='Done!')
    return annotated, f'**Completed in {elapsed:.1f}s**\n\n' + summary, pdf_path


# ════════════════════════════════════════════════════════════
#  8. GRADIO UI
# ════════════════════════════════════════════════════════════
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Sora:wght@300;400;600;700&display=swap');
:root { --navy:#0D2137; --accent:#2E86C1; --bg:#F2F6FB; --muted:#6B7FA3; }
body, .gradio-container { font-family:'Sora',sans-serif !important; background:var(--bg) !important; }
#dg-hdr { background:linear-gradient(135deg,#0D2137 0%,#1A5276 55%,#2E86C1 100%);
          border-radius:14px; padding:24px 32px 18px; margin-bottom:16px;
          box-shadow:0 8px 32px rgba(13,33,55,.20); }
#dg-hdr h1 { color:#fff; font-size:2.0em; font-weight:700; margin:0 0 4px; letter-spacing:-.5px; }
#dg-hdr p  { color:rgba(255,255,255,.72); font-size:.9em; margin:0 0 10px; font-weight:300; }
.badge { display:inline-block; background:rgba(255,255,255,.13);
         border:1px solid rgba(255,255,255,.22); border-radius:20px;
         padding:2px 11px; font-size:.72em; color:rgba(255,255,255,.88); margin-right:5px; }
label span { font-family:'Sora',sans-serif !important; font-weight:600 !important;
             font-size:.83em !important; color:var(--navy) !important;
             text-transform:uppercase; letter-spacing:.4px; }
#gen-btn { background:linear-gradient(135deg,#0D2137 0%,#2E86C1 100%) !important;
           color:white !important; font-weight:700 !important;
           border-radius:10px !important; border:none !important;
           box-shadow:0 4px 16px rgba(46,134,193,.38) !important; transition:all .2s ease !important; }
#gen-btn:hover { transform:translateY(-2px) !important; }
#demo-btn  { border:1.5px solid #2E86C1 !important; color:#2E86C1 !important;
             background:transparent !important; border-radius:10px !important; font-weight:600 !important; }
#clear-btn { border:1.5px solid var(--muted) !important; color:var(--muted) !important;
             background:transparent !important; border-radius:10px !important; }
.footer { text-align:center; color:var(--muted); font-size:.77em; margin-top:14px; }
"""

with gr.Blocks(css=CSS, title='Doc-Genie v2.0', theme=gr.themes.Base()) as demo:

    gr.HTML("""
    <div id="dg-hdr">
      <h1>&#129670; Doc-Genie v2.0</h1>
      <p>Professional Python Docstring Generator &mdash; IBM Granite 3.3 2B Instruct &times; AST Analysis</p>
      <span class="badge">&#129302; IBM Granite 3.3 2B</span>
      <span class="badge">&#128013; AST Analysis</span>
      <span class="badge">&#128214; Google &amp; NumPy Styles</span>
      <span class="badge">&#128196; PDF Export</span>
      <span class="badge">&#129303; HuggingFace</span>
    </div>""")

    with gr.Row(equal_height=False):
        with gr.Column(scale=5):
            code_input = gr.Code(language='python', label='Input Python Code',
                                 value=DEMO_CODE, lines=28)
            with gr.Row():
                style_dd = gr.Dropdown(['Google','NumPy'], value='Google',
                                       label='Docstring Style', scale=2)
                max_tok  = gr.Slider(150, 700, value=450, step=25,
                                     label='Max Tokens / Function', scale=3)
            with gr.Accordion('Advanced Settings', open=False):
                temp = gr.Slider(0.1, 1.0, value=0.35, step=0.05,
                                 label='Temperature (lower = more deterministic)')
            with gr.Row():
                demo_btn  = gr.Button('Load Demo',             elem_id='demo-btn',  scale=1)
                clear_btn = gr.Button('Clear',                 elem_id='clear-btn', scale=1)
                gen_btn   = gr.Button('Generate Docstrings', elem_id='gen-btn',   scale=3)

        with gr.Column(scale=5):
            with gr.Tabs():
                with gr.Tab('Annotated Code'):
                    code_out = gr.Code(language='python', label='Annotated Output',
                                       lines=28, interactive=False)
                with gr.Tab('Analysis Report'):
                    status   = gr.Markdown('*Run Doc-Genie to see the analysis here.*')
            pdf_file = gr.File(label='Download PDF Report', interactive=False)

    with gr.Accordion('How Doc-Genie Works', open=False):
        gr.Markdown("""
| Step | Component | What Happens |
|------|-----------|--------------|
| 1 | **ASTAnalyser** | Parses source with Python `ast`; extracts types, defaults, raises, loops, I/O, complexity |
| 2 | **PromptBuilder** | Builds a structured expert prompt using Granite chat format |
| 3 | **ModelManager** | Sends prompt to IBM Granite 3.3 2B via HuggingFace Transformers |
| 4 | **DocstringGenerator** | Cleans output; injects docstring after `def` via regex |
| 5 | **PDFExporter** | Builds professional PDF with metadata tables and annotated source |
""")

    gr.HTML('<div class="footer">Doc-Genie v2.0 &middot; Python &middot; AST &middot; Gradio &middot; IBM Granite 3.3 2B &middot; HuggingFace &middot; ReportLab</div>')

    demo_btn.click(fn=lambda: DEMO_CODE, outputs=code_input)
    clear_btn.click(fn=lambda: ('', '*Cleared.*', None),
                    outputs=[code_out, status, pdf_file]
                    ).then(fn=lambda: '', outputs=code_input)
    gen_btn.click(fn=run_docgenie,
                  inputs=[code_input, style_dd, max_tok, temp],
                  outputs=[code_out, status, pdf_file])


# ════════════════════════════════════════════════════════════
#  9. LAUNCH
# ════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('  Doc-Genie v2.0 launching...')
print('  Open the Gradio URL printed below in your browser.')
print('='*60 + '\n')
demo.launch(share=True, debug=False, show_error=True, inbrowser=False)